# DEV - Librarian

**Tasks**
- ⬜ `if __name__ == "__main__":` to be used from command line with `sys.argv`
- ⬜ `#f` skip `!filepath`s also when copying;
- ⬜ `#u` final clean: search empty local folder and delete

```python
    dirname = 'a'

    if not os.listdir(dirname):
        os.rmdir(dirname)
        print(f"{dirname} removed!")
```

**Walk Recursively**

```python
    from pathlib import Path
    import hashlib
    import os


    def walk(path, _basepath=None): 
        if _basepath is None:
            _basepath=path
        
        for p in Path(path).iterdir(): 
            if p.name.startswith('.'):
                continue
            if p.is_dir(): 
                yield from walk(p, _basepath=_basepath)
                continue
            if not os.path.isfile(p) or os.path.islink(p):
                continue

            nn = p.resolve()
            # PermissionError...
            with open(nn, 'rb') as ff:
                yield nn.relative_to(_basepath).as_posix(), hashlib.md5(ff.read()).hexdigest()
```

- ✅ `#a` filter on files/folder to be included (instead of excluded)
- ✅ `#i` progress bar when **adding/deleting/modifying** files: `tqdm` applied to each adding/deleting/modifying loop, actions/errors no printout, log only
- ✅ `#t` save log test
- ✅ `if __name__ == "__main__":` to be used while launching the script
- ✅ `#f` folder/path skip overlapping
- ✅ `#a` folders/files to be excluded from ext folder (no **add** neither **modify**)
- ✅ `#a` folders/files to be skipped in local folder (no **delete** neither **modify**)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append('..')
import shutil
from librarian import Librarian

TEST_EXT_FOLDER = './test_ext_folder'
TEST_LOCAL_FOLDER = './test_local_folder'

In [15]:
os.makedirs(TEST_EXT_FOLDER, exist_ok=True)
if os.path.exists(TEST_LOCAL_FOLDER):
    shutil.rmtree(TEST_LOCAL_FOLDER)   # delete test folder
librarian = Librarian(ext_path=TEST_EXT_FOLDER, 
                      local_path=TEST_LOCAL_FOLDER,
                      libignore_filepath=".libignore",
                      update=False,
                      filtering=True)

Local path created: C:\Users\i.stiaccini\Desktop\librarian\dev\test_local_folder
Scanning EXTERNAL folder ... done!
Scanning LOCAL folder ... done!
Comparing folders ... done!


In [32]:
if os.path.exists(TEST_LOCAL_FOLDER):
    shutil.rmtree(TEST_LOCAL_FOLDER)   # delete test folder
librarian.update_folder_content()
librarian.compare_folders()
librarian.update_local_folder(log=False, filtering=True)

Scanning EXTERNAL folder ... done!
Scanning LOCAL folder ... done!
Comparing folders ... done!
Local path created: C:\Users\i.stiaccini\Desktop\librarian\dev\test_local_folder


Copy: 100%|█| 5.00/5.00 [00:00<00:00, 1.25kstep/s]


  boh.md
  b\test.md


Delete:   0%|       | 0.00/5.00 [00:00<?, ?step/s]
Modify:   0%|       | 0.00/5.00 [00:00<?, ?step/s]

--> Completed!!


In [15]:
librarian.update_folder_content()
librarian.compare_folders()
librarian.added

Scanning EXTERNAL folder ... done!
Scanning LOCAL folder ... done!
Comparing folders ... done!


{WindowsPath('a/aa.txt'),
 WindowsPath('boh.md'),
 WindowsPath('cc.txt'),
 WindowsPath('readme.txt')}

# Testing `match` Function

In [ ]:
import fnmatch

def match_pattern(path, pattern, base_dir):
    " Match a path to a .gitignore-style pattern. "
    # Normalize
    path = path.replace(os.sep, '/')
    base_dir = base_dir.replace(os.sep, '/')
    pattern = pattern.replace(os.sep, '/')

    # If pattern starts with '/', it's relative to the base directory
    if pattern.startswith('/'):
        pattern = pattern[1:]
        path_rel = os.path.relpath(path, base_dir).replace(os.sep, '/')
    else:
        path_rel = path

    # Convert ** to */*/ and handle directory endings
    if pattern.endswith('/'):
        pattern = pattern.rstrip('/') + '/**'

    # Convert gitignore wildcards to fnmatch wildcards
    # Git ** matches across directories
    pattern = pattern.replace('**', '*')

    return fnmatch.fnmatch(path_rel, pattern)

## `pathlib` Notes

In [39]:
from pathlib import Path
filepath = Path(TEST_EXT_FOLDER).resolve() / "e.txt"
print("Full Filepath:", filepath, type(filepath))
print("Filepath:", filepath.__str__(), type(filepath.__str__()))
print("Full Folder Path:", filepath.parent)
print("Parent Folder only:", filepath.parent.name)
print("All Parent Folders:", filepath.parents[:])
print("Filename:", filepath.name, type(filepath.name))


Full Filepath: C:\Users\isax7\Desktop\librarian\dev\test_ext_folder\e.txt <class 'pathlib.WindowsPath'>
Filepath: C:\Users\isax7\Desktop\librarian\dev\test_ext_folder\e.txt <class 'str'>
Full Folder Path: C:\Users\isax7\Desktop\librarian\dev\test_ext_folder
Parent Folder only: test_ext_folder
All Parent Folders: (WindowsPath('C:/Users/isax7/Desktop/librarian/dev/test_ext_folder'), WindowsPath('C:/Users/isax7/Desktop/librarian/dev'), WindowsPath('C:/Users/isax7/Desktop/librarian'), WindowsPath('C:/Users/isax7/Desktop'), WindowsPath('C:/Users/isax7'), WindowsPath('C:/Users'), WindowsPath('C:/'))
Filename: e.txt <class 'str'>


## Folder Error Check

In [45]:
TEST_EXT_FOLDER_ERROR = TEST_EXT_FOLDER + "_error"
librarian = Librarian(ext_path=TEST_EXT_FOLDER_ERROR, 
                    local_path=TEST_LOCAL_FOLDER,
                    update=False)

ValueError: Folder ./test_ext_folder_error does not exists!